In [ ]:
import numpy as np
from skopt import gp_minimize
from skopt.space import Real, Integer, Categorical
from AIAA_DBF_2526 import RCPlane
import pygad
import pyswarms as ps
import matplotlib.pyplot as plt
from pyswarms.utils.plotters import plot_cost_history
import logging 

## Mission Scoring Functions

In [ ]:
def round_inches(x: float) -> float:
    """
    Round a measurement in inches down to the nearest 0.00, 0.25, 0.50, or 0.75.

    Example:
        10.37 -> 10.25
        5.81  -> 5.75
        7.99  -> 7.75
    """
    # Whole inch part
    whole = int(x)

    # Fractional part
    frac = x - whole

    # Define breakpoints
    targets = [0.00, 0.25, 0.50, 0.75]

    # Find the largest target <= frac
    rounded_frac = max([t for t in targets if t <= frac], default=0.00)

    return whole + rounded_frac

def mission_2(num_passengers, num_cargo, m2_laps, battery_capacity):

    income = (num_passengers * (6 + 2 * m2_laps)) + (num_cargo * (10 + 8 * m2_laps))
    EF = battery_capacity / 100
    cost = m2_laps * (10 + (num_passengers*0.5) + (num_cargo*2)) * EF
    net_income = income - cost
    m2 = 1 + (net_income / 1940)
    return m2

def mission_3(banner_length, number_of_laps, wing_span):
    """
    :param banner_length: length in inches
    :param number_of_laps: number of laps
    :param wing_span: span in metres
    :return: mission 3 score with best score of 410
    """
    wing_span_inches = wing_span * 39.3701
    pre = round(wing_span_inches)/12 * 0.05 + 0.75
    RAC = pre if pre >= 0.9 else 0.9
    rounded_banner = round_inches(banner_length)
    m3 = (rounded_banner * number_of_laps / RAC)
    m3_best = 1200
    return 2 + (m3 / m3_best)

def mission_2raw(num_passengers, num_cargo, m2_laps, battery_capacity):

    income = (num_passengers * (6 + 2 * m2_laps)) + (num_cargo * (10 + 8 * m2_laps))
    EF = battery_capacity / 100
    cost = m2_laps * (10 + (num_passengers*0.5) + (num_cargo*2)) * EF
    net_income = income - cost
    return net_income

def mission_3raw(banner_length, number_of_laps, wing_span):
    """
    :param banner_length: length in inches
    :param number_of_laps: number of laps
    :param wing_span: span in metres
    :return: mission 3 score with best score of 410
    """
    wing_span_inches = wing_span * 39.3701
    pre = round(wing_span_inches)/12 * 0.05 + 0.75
    RAC = pre if pre >= 0.9 else 0.9
    rounded_banner = round_inches(banner_length)
    m3 = (rounded_banner * number_of_laps / RAC)
    return m3

gm_raw = lambda x, y: 1.8 * (x + y) + 25

gm = lambda x, y : 32.2 / gm_raw(x, y)

In [ ]:
import math

def pretty_print_plane(plane):
    """Prints a nicely formatted summary of the RCPlane dataclass."""
    
    print("=" * 55)
    print(" 🛩️  AIAA DBF 25/26 - AIRCRAFT DESIGN SUMMARY")
    print("=" * 55)

    # --- SYSTEM OVERVIEW ---
    print("\n[ SYSTEM CONVERGENCE ]")
    status = "✅ PASS" if plane.is_converged else "❌ FAIL"
    print(f"  Target Struct Mass : {plane.m_struct:.3f} kg")
    print(f"  Convergence Status : {status} (Error: {plane.relative_error * 100:.3f}%)")

    # --- AEROSTRUCTURES ---
    print("\n[ AEROSTRUCTURES ]")
    # Wing
    print(f"  Wing:")
    print(f"    - Airfoil        : {plane.wing.airfoil_type}")
    print(f"    - Dimensions     : Span = {plane.wing.span:.2f} m | Chord = {plane.wing.chord:.3f} m")
    print(f"    - Aspect Ratio   : {plane.wing.aspect_ratio:.2f}  | Area  = {plane.wing.surface_area:.3f} m²")
    print(f"    - Mass           : {plane.wing.mass:.3f} kg")
    # Spar
    print(f"  Main Spar:")
    print(f"    - Material       : {plane.wing.spar.material} ({'Sized' if plane.wing.spar.is_sized else 'Unsized'})")
    print(f"    - Dimensions     : OD = {plane.wing.spar.outer_diameter * 1000:.1f} mm | Wall = {plane.wing.spar.wall_thickness * 1000:.1f} mm")
    print(f"    - Mass           : {plane.wing.spar.mass:.3f} kg")
    # Tail
    print(f"  Empennage (Inverted T-Tail):")
    print(f"    - H-Tail         : Span = {plane.tail.span_H:.2f} m | Area = {plane.tail.area_H:.3f} m²")
    print(f"    - V-Tail         : Span = {plane.tail.span_V:.2f} m | Area = {plane.tail.area_V:.3f} m²")
    print(f"    - Mass           : {plane.tail.mass:.3f} kg")
    # Fuselage
    print(f"  Fuselage:")
    print(f"    - Mass           : {plane.fuselage.mass:.3f} kg")
    print(f"    - Wing AC Pos.   : {plane.fuselage.wing_ac_from_nose:.3f} m from nose")

    # --- PROPULSION ---
    print("\n[ PROPULSION & POWER ]")
    print(f"  - Motor Max Power  : {plane.propulsion.motor_power:.1f} W")
    print(f"  - Effective Power  : {plane.propulsion.effective_power:.1f} W (Efficiencies: Motor {plane.propulsion.motor_eff*100:.0f}% / Prop {plane.propulsion.prop_eff*100:.0f}%)")
    print(f"  - Propulsion Mass  : {plane.propulsion.mass:.3f} kg")

    # --- MISSIONS ---
    print("\n" + "=" * 55)
    print(" 📋 FLIGHT MISSIONS ANALYSIS")
    print("=" * 55)

    for m_id, mission in plane.missions.items():
        # Bank angle is likely in radians, converting to degrees for readability
        bank_deg = math.degrees(mission.performance.bank_angle)
        
        print(f"\n>>> MISSION {m_id} <<<")
        print(f"  Payload & Weights:")
        print(f"    - Payload Mass   : {mission.payload:.3f} kg")
        print(f"    - Battery Mass   : {mission.avionics.mass_battery:.3f} kg (Capacity: {mission.avionics.capacity:.1f})")
        
        print(f"  Flight Performance:")
        print(f"    - Speeds         : Cruise = {mission.performance.V_cruise:.2f} m/s | Stall = {mission.performance.V_stall:.2f} m/s")
        print(f"    - Aero           : Cruise CL = {mission.performance.CL_cruise:.3f} | L/D = {mission.performance.L_D_cruise:.2f}")
        print(f"    - Safety Margins : Stall = {mission.performance.stall_margin:.2f}x | Load Factor = {mission.performance.load_factor_margin:.2f}x")
        
        print(f"  Maneuvering (Turn):")
        print(f"    - Turn Radius    : {mission.performance.turn_radius:.2f} m at {mission.performance.v_turn:.2f} m/s")
        print(f"    - Bank Angle     : {bank_deg:.1f}°")
        print(f"    - Load Factor (n): {mission.performance.n_turn:.2f} g (Max: {mission.performance.n_max:.2f} g)")
        
        print(f"  Endurance:")
        print(f"    - Est. Time      : {mission.performance.flight_time:.1f} sec")
        print(f"    - Laps in 5 mins : {mission.performance.number_of_laps:.0f} laps")
        print("-" * 55)

In [ ]:
import pandas as pd
import time

class OptimizationTracker:
    def __init__(self, algorithm_name):
        self.algo_name = algorithm_name
        self.history = []
        self.start_time = time.time()
        
    def log_evaluation(self, x, plane, final_score, bonus_score, error_msg=""):
        # Unpack the decision variables
        m_struct, wing_span, motor_power, wing_AR, n_pucks, pax_cargo_ratio, m1_batt, m2_batt, banner_len, banner_AR, m3_batt = x
        
        # Base record with inputs
        record = {
            'iteration': len(self.history) + 1,
            'time_elapsed_sec': time.time() - self.start_time,
            'm_struct': m_struct, 'wing_span': wing_span, 'motor_power': motor_power,
            'wing_AR': wing_AR, 'n_pucks': n_pucks, 'pax_cargo_ratio': pax_cargo_ratio,
            'm1_battery': m1_batt, 'm2_battery': m2_batt, 
            'banner_length': banner_len, 'banner_AR': banner_AR, 'm3_battery': m3_batt,
            'error_msg': error_msg,
            'final_score': final_score,
            'bonus_score': bonus_score,
            'total_score': final_score + bonus_score
            
        }
        
        # If the plane successfully initialized without throwing physical constraint errors
        if plane:
            record.update({
                'is_converged': plane.is_converged,
                'relative_error': plane.relative_error,
                'm_max': plane.m_max,
                'wing_loading_N_m2': plane.wing_loading,
                'power_to_weight_W_N': plane.power_to_weight,
                'gm_score': plane.missions.get("gm", 0)
            })
            
            # 3. Dynamically extract Performance metrics for each mission
            for m_idx in ["1", "2", "3"]:
                if m_idx in plane.missions:
                    mission = plane.missions[m_idx]
                    perf = mission.performance
                    prefix = f"m{m_idx}_"
                    
                    record.update({
                        f"{prefix}score": mission.score,
                        f"{prefix}payload_mass": mission.payload,
                        f"{prefix}battery_mass": mission.avionics.mass_battery,
                        # Aerodynamic Performance
                        f"{prefix}CL_cruise": perf.CL_cruise,
                        f"{prefix}V_cruise": perf.V_cruise,
                        f"{prefix}V_stall": perf.V_stall,
                        # Standard Turn Performance
                        f"{prefix}n_turn": perf.n_turn,
                        f"{prefix}v_turn": perf.v_turn,
                        f"{prefix}turn_radius": perf.turn_radius,
                        f"{prefix}bank_angle_rad": perf.bank_angle,
                        # Maximum Limits
                        f"{prefix}n_max": perf.n_max,
                        f"{prefix}max_v_turn": perf.max_v_turn,
                        f"{prefix}max_bank_angle_rad": perf.max_bank_angle,
                        f"{prefix}max_turn_radius": perf.max_turn_radius,
                        # Flight Lap Performance
                        f"{prefix}flight_time": perf.flight_time,
                        f"{prefix}number_of_laps": perf.number_of_laps,
                        # Metrics
                        f"{prefix}L_D": perf.L_D_cruise,
                        f"{prefix}stall_margin": perf.stall_margin,
                        f"{prefix}load_factor_margin": perf.load_factor_margin
                    })
        else:
            # Default empty values for failed planes
            record.update({
                'is_converged': False, 'relative_error': None, 'm_max': None,
                'wing_loading_N_m2': None,
                'power_to_weight_W_N': None,
            })
            
        self.history.append(record)

    def save_to_csv(self):
        df = pd.DataFrame(self.history)
        filename = f"{self.algo_name}_optimization_history.csv"
        df.to_csv(filename, index=False)
        print(f"Saved {len(self.history)} iterations to {filename}, Time taken: {self.history[-1]['time_elapsed_sec']}")
        return df
    

## Combined Mission Objective Function 

In [57]:
def engineering_bonus(plane):
    # Reward higher L/D
    ld_bonus = 0.1 * plane.missions["1"].performance.L_D_cruise
    
    # Reward a safe stall margin (aiming for 1.25)
    stall_val = plane.missions["1"].performance.stall_margin
    stall_bonus = -abs(stall_val - 1.3) * 0.5
    # Reward higher AR but with a 'diminishing return'+-
    # This prevents the optimizer from going to AR=50
    ar_bonus = 0.05 * plane.wing.aspect_ratio if plane.wing.aspect_ratio < 10 else 0
    # Penalise sizing flight time to within 5 minutes. 
    ft_penalty = 0
    idx = ["1", "2", "3"]
    for i in idx:
        flight_time = plane.missions[f"{i}"].performance.flight_time
        ft_penalty -= 2 + (0.1 * (flight_time - 335)) if flight_time > 335 else 0 #335 for leeway of penalty
    return ld_bonus + stall_bonus + ar_bonus + ft_penalty

def objective_tracked(x, tracker):
    """
    Calculates the combined (negative) score for Mission 2 and Mission 3.

    Args:
        x (tuple/array): Decision variables (m_struct, wing_span, motor_power, wing_AR, n_pucks, 
            passenger_cargo_ratio, m2_battery, banner_length, banner_AR, m3_battery).
        tracker: to track the variables
    """
    # Unpack decision variables
    m_struct, wing_span, motor_power, wing_AR, n_pucks, passenger_cargo_ratio, m1_battery, m2_battery, banner_length, banner_AR, m3_battery = x
    
    
    # Scoring
    try:
        plane = RCPlane(m_struct=m_struct, wing_span=wing_span, wing_AR=wing_AR, motor_power=motor_power, m1_battery=m1_battery, n_pucks=n_pucks, passenger_cargo_ratio=passenger_cargo_ratio, m2_battery=m2_battery, banner_length=banner_length, banner_AR=banner_AR, m3_battery=m3_battery)

        m1_score = 1 if plane.missions["1"].performance.number_of_laps > 2 else 0
        m2_score = mission_2(n_pucks * passenger_cargo_ratio, n_pucks, plane.missions["2"].performance.number_of_laps, m2_battery)
        m3_score = mission_3(banner_length, plane.missions["3"].performance.number_of_laps, wing_span)
        gm_score = gm(n_pucks * passenger_cargo_ratio, n_pucks)
    
        plane.missions["1"].update_score(m1_score)
        plane.missions["2"].update_score(m2_score)
        plane.missions["3"].update_score(m3_score)
        plane.missions["gm"] = gm_score
        
        if not m1_score: # if failed mission 1, return 0
            final_score = 0
            error_msg = "M1 Failed"
        elif not m2_score: # if failed mission 2, return m1 score only
            final_score = m1_score
            error_msg = "M2 Failed"
        elif not m3_score: # if failed mission 3, return m1 + m2 score only
            final_score = m1_score + m2_score
            error_msg = "M3 Failed"
        else:
            final_score = m1_score + m2_score + m3_score + gm_score
            error_msg = ""
        
        bonus = engineering_bonus(plane)
        rel_error = plane.relative_error
        convergence_penalty = (rel_error * 1000)
        total_performance = final_score + bonus
        if not plane.is_converged:
            tracker.log_evaluation(x, None, -1, 0, error_msg="Mass not Coherent")
            return -convergence_penalty + (total_performance * 0.1)
        else:
                            # Log the successful evaluation
            tracker.log_evaluation(x, plane, final_score, bonus, error_msg=error_msg)
            if final_score > 0: 
                logging.info(f"\nM1: {m1_score}\nM2: {m2_score}\nM3: {m3_score}\nGM: {gm_score}\nFINAL SCORE: {final_score}\nBonus: {bonus}")
            return -convergence_penalty + (total_performance * 10)

    except Exception as error:
        #print(f"RCPlane Error: {error}")
        tracker.log_evaluation(x, None, -1, 0, error_msg=str(error))
        # Physical failure (crash, N_max error, etc.)
        return -100 + (rel_error * 100 if 'rel_error' in locals() else 50)
    


In [ ]:
import itertools

start = time.time()
bf_tracker = OptimizationTracker("Brute_Force")

# Define very coarse steps for Brute Force to keep compute time reasonable
grid_spaces = [
    np.linspace(0.5, 5, 3),      # m_struct
    np.linspace(1.0, 1.5, 2),    # wing_span
    np.linspace(200, 1000, 2),   # motor_power
    [4, 6, 8],                   # wing_AR
    [10, 20],                    # n_pucks
    [5, 10],                     # passenger_cargo_ratio
    [10, 20, 60],                    # m1_battery
    [30, 40, 60],                    # m2_battery
    [200, 500],                  # banner_length
    [2, 4],                      # banner_AR
    [30, 50, 60]                     # m3_battery
]

print("Starting Brute Force... this may take a while.")
for combination in itertools.product(*grid_spaces):
    # Call the objective function directly (we ignore the return value since tracker logs it)
    objective_tracked(combination, bf_tracker)

bf_df = bf_tracker.save_to_csv()

print("timetaken:", time.time() - start)

In [ ]:
pso_tracker = OptimizationTracker("PSO")

# Wrapper for PySwarms which passes multiple particles at once
def pso_objective_wrapper(particles):
    n_particles = particles.shape[0]
    scores = np.zeros(n_particles)
    for i in range(n_particles):
        rounded_particle = particles[i]
        # Round the discrete variables by their index:
        rounded_particle[2] = np.round(rounded_particle[2])               # Index 2: motor power
        rounded_particle[3] = np.round(rounded_particle[3] * 100) / 100   # Index 3: wing AR to 3.9X
        rounded_particle[4] = np.round(rounded_particle[4])               # Index 4: n_pucks
        rounded_particle[5] = np.round(rounded_particle[5])               # Index 5: pax_cargo_ratio
        rounded_particle[9] = np.round(rounded_particle[9]* 100) / 100    # Index 9: Banner AR to 3.9X
        rounded_particle[10] = np.round(rounded_particle[10])             # Index 10: m3 batt
        scores[i] = -objective_tracked(particles[i], pso_tracker)
    return scores

# Define boundaries: [m_struct, wing_span, motor_power, wing_AR, n_pucks, pax_cargo_ratio, m1_batt, m2_batt, banner_len, banner_AR, m3_batt]
lower_bounds = np.array([0.2, 0.92, 40, 1, 1, 3, 1, 1, 10, 1, 1])
upper_bounds = np.array([20, 1.52, 4000, 8, 30, 15, 100, 100, 1000, 5, 100])
bounds = (lower_bounds, upper_bounds)


In [ ]:
def PSO_objective_f(x):
    # Unpack PSO parameters
    c1, c2, w = x
    
    # Define boundaries: [m_struct, wing_span, motor_power, wing_AR, n_pucks, pax_cargo_ratio, m1_batt, m2_batt, banner_len, banner_AR, m3_batt]
    
    # Initialize swarm parameters
    options = {'c1': c1, 'c2': c2, 'w': w} # Cognitive, Social, Inertia parameters
    
    optimizer = ps.single.GlobalBestPSO(n_particles=50, dimensions=11, options=options, bounds=bounds)
    best_cost, best_pos = optimizer.optimize(pso_objective_wrapper, iters=250)
    
    # 3. Get the score of the best plane found (maximize score, so it's negative fitness)
    # 5. Get the best solution
        
    # BO minimizes, so we want the most negative average fitness
    return best_cost


# sol_per_pop, crossover_probability, mutation_probability, K_tournament
search_space = [
    Real(0.5, 2.5, name="c1"),
    Real(0.5, 2.5, name="c2"),
    Real(0.4, 0.9, name="w")
]

# Run Bayesian Optimization
result = gp_minimize(
    PSO_objective_f,
    search_space,
    n_calls=30,
    n_initial_points=10,
    random_state=42
)


# Best design
best_params = result.x
print(best_params)

from skopt.plots import plot_convergence
plot_convergence(result)

In [ ]:
# Initialize swarm parameters
# {'c1': 1.6937003158929742, 'c2': 1.3916655057071825, 'w': 0.44998745790900146}
# [1.8597085402784266, 0.6764254336947868, 0.8063693437585324]
# [1.738336075497409, 1.3969107799046967, 0.4500706027319868] best
# [1.7612994689242707, 0.6028815218081174, 0.6370112367098811]
c1, c2, w = [1.738336075497409, 1.3969107799046967, 0.4500706027319868]
options = {'c1': c1, 'c2': c2, 'w': w} # Cognitive, Social, Inertia parameters

optimizer = ps.single.GlobalBestPSO(n_particles=250, dimensions=11, options=options, bounds=bounds)
best_cost, best_pos = optimizer.optimize(pso_objective_wrapper, iters=1250)

# Save results
pso_df = pso_tracker.save_to_csv()

# Visualize the convergence
plot_cost_history(cost_history=optimizer.cost_history)
plt.show()

In [ ]:
m_struct, wing_span, motor_power, wing_AR, n_pucks, passenger_cargo_ratio, m1_battery, m2_battery, banner_length, banner_AR, m3_battery = best_pos
plane = RCPlane(m_struct=m_struct, wing_span=wing_span, wing_AR=wing_AR, motor_power=motor_power, m1_battery=m1_battery, n_pucks=n_pucks, passenger_cargo_ratio=passenger_cargo_ratio, m2_battery=m2_battery, banner_length=banner_length, banner_AR=banner_AR, m3_battery=m3_battery)
pretty_print_plane(plane)


## Genetic Algo

In [ ]:
# 1. Define the Range and Type for each Parameter (Gene)
gene_space = [
    # m_struct (Real: 0.2 to 20)
    {'low': 0.2, 'high': 13},
    
    # wing_span (Real: 0.92 to 1.52)
    {'low': 0.92, 'high': 1.52, 'step': 0.01},
    
    # motor_power (Integer: 40 to 4000)
    {'low': 40, 'high': 4000, 'step': 10},
    
    # wing_AR (Integer: 3 to 8)
    {'low': 3, 'high': 8, 'step': 0.1},
    
    # n_pucks (Integer: 1 to 30)
    {'low': 1, 'high': 30, 'step': 1},
    
    # passenger_cargo_ratio (Integer: 3 to 15)
    {'low': 3, 'high': 15, 'step': 1},
    
    # m1_battery (Integer/Real Wh: 1 to 100)
    {'low': 1, 'high': 100, 'step': 5}, 
    
    # m2_battery (Integer/Real Wh: 1 to 100)
    {'low': 1, 'high': 100, 'step': 5}, 
    
    # banner_length (Integer: 1 to 1000)
    {'low': 10, 'high': 1000, 'step': 1},
    
    # banner_AR (Integer: 1 to 5)
    {'low': 1, 'high': 5, 'step': 0.1},
    
    # m3_battery (Integer/Real Wh: 1 to 100)
    {'low': 1, 'high': 100, 'step': 5}
]
GA_tracker = OptimizationTracker("GA")
def fitness_func(ga_instance, solution, solution_idx):
    fitness = objective_tracked(solution, GA_tracker)
    return fitness

fitness_function = fitness_func

### Bayesian Optimisation for Genetic Algorithm Hyperparameters

In [ ]:
def GA_objective_f(x):
    # Unpack GA parameters: sol_per_pop, crossover_probability, mutation_probability, ...
    sol_per_pop, crossover_probability, mutation_probability, K_tournament = x
    sol_per_pop = int(sol_per_pop)
    
    # 1. Initialize PyGAD with the input ga_hyperparameters
    ga_instance = pygad.GA(
        num_generations=50,             # Number of iterations
        num_parents_mating=20,          # Number of solutions to select as parents
        fitness_func=fitness_func,      # The function to maximize
        sol_per_pop=sol_per_pop,                 # Number of solutions in the population
        num_genes=11,                   # Total number of parameters
        gene_space=gene_space,          # <-- This tells PyGAD the bounds!
        mutation_type="random",
        mutation_percent_genes=10,      # Percentage of genes to mutate
        crossover_type="single_point",
        parent_selection_type="tournament",# Steady-State Selection
        K_tournament=K_tournament,
        keep_elitism=2,
        mutation_probability=mutation_probability,
        crossover_probability=crossover_probability
    )
    
    # 2. Run the inner GA to find the best plane
    ga_instance.run()
    
    # 3. Get the score of the best plane found (maximize score, so it's negative fitness)
    # 5. Get the best solution
    solution, solution_fitness, solution_idx = ga_instance.best_solution()
        
    # BO minimizes, so we want the most negative average fitness
    return -solution_fitness


# sol_per_pop, crossover_probability, mutation_probability, K_tournament
search_space = [
    Integer(100, 500, name="sol_per_pop"),
    Categorical([0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0], name="crossover_probability"),
    Real(0.01, 0.2, name="mutation_probability"),
    Integer(3, 7, name="K_tournament")
]

# Run Bayesian Optimization
result = gp_minimize(
    GA_objective_f,
    search_space,
    n_calls=40,
    n_initial_points=10,
    random_state=42
)

# Best design
best_params = result.x
print(best_params)

from skopt.plots import plot_convergence
plot_convergence(result)

[450, 0.65, 0.15814129005182623, np.int64(5)]
[np.int64(200), np.float64(0.8), 0.2, np.int64(7)]

In [ ]:
sol_per_pop, crossover_probability, mutation_probability, K_tournament = [450, 0.65, 0.15814129005182623, np.int64(5)]
start = time.time()
# 3. Initialize PyGAD
ga_instance = pygad.GA(
    num_generations=250,             # Number of iterations
    num_parents_mating=20,          # Number of solutions to select as parents
    fitness_func=fitness_func,      # The function to maximize
    sol_per_pop=sol_per_pop,                 # Number of solutions in the population
    num_genes=11,                   # Total number of parameters
    gene_space=gene_space,          # <-- This tells PyGAD the bounds!
    mutation_type="random",
    mutation_percent_genes=10,      # Percentage of genes to mutate
    crossover_type="single_point",
    parent_selection_type="tournament",# Steady-State Selection
    K_tournament=K_tournament,
    keep_elitism=2,
    mutation_probability=mutation_probability,
    crossover_probability=crossover_probability,
    # save_solutions=True
)
# 4. Run the optimization
ga_instance.run()

# Optional: Plot the fitness progression
ga_instance.plot_fitness()

print(time.time() - start)

In [ ]:
# 5. Get the best solution
solution, solution_fitness, solution_idx = ga_instance.best_solution()

# Get the best inputs, ensuring integers are rounded and cast
best_params = solution
# best_params[2] = int(round(best_params[2]))   # motor_power
# best_params[3] = int(round(best_params[3]))   # wing_AR
# best_params[4] = int(round(best_params[4]))   # n_pucks
# best_params[5] = int(round(best_params[5]))   # P/C_ratio
# best_params[8] = int(round(best_params[8]))   # banner_length
# best_params[9] = int(round(best_params[9]))   # banner_AR

param_names = [
    "m_struct (kg)", "wing_span (m)", "motor_power (W)", "wing_AR", "n_pucks", 
    "P/C_ratio", "m1_battery (Wh)", "m2_battery (Wh)", "banner_length (in)", "banner_AR", "m3_battery (Wh)"
]

print(f"Best Fitness Score: {solution_fitness:.4f}")
print(f"Best Solution found (Gene Array): {best_params}")

# Display in a readable format
print("\n--- Best RCPlane Configuration ---")
# Define a consistent alignment width (e.g., 10 characters)
VALUE_WIDTH = 10 

for name, value in zip(param_names, best_params):
    # Use appropriate formatting based on parameter type
    
    # If the parameter is Real/Mass/Span, use 3 decimal places and right alignment
    if 'Real' in name or 'mass' in name or 'span' in name:
        format_str = f">{VALUE_WIDTH}.3f" # Example: >10.3f
    
    # If the parameter is Integer, use right alignment with no decimal
    else:
        format_str = f">{VALUE_WIDTH}" # Example: >10

    # Print the line using the fully constructed format string
    print(f"  {name:<25}: {value:{format_str}}")

# Optional: Run the best solution through the RCPlane class again to verify full performance metrics
# This step is good practice as the fitness function only returned the score.

m_struct, wing_span, motor_power, wing_AR, n_pucks, passenger_cargo_ratio, m1_battery, m2_battery, banner_length, banner_AR, m3_battery = best_params

best_plane = RCPlane(m_struct=m_struct, wing_span=wing_span, wing_AR=wing_AR, motor_power=motor_power, m1_battery=m1_battery, n_pucks=n_pucks, passenger_cargo_ratio=passenger_cargo_ratio, m2_battery=m2_battery, banner_length=banner_length, banner_AR=banner_AR, m3_battery=m3_battery)

pretty_print_plane(best_plane)



## Particle Swarm Optimisation

In [ ]:
GA_tracker.save_to_csv()

In [ ]:
solutions_df = pd.read_csv("GA_optimization_history.csv")

In [ ]:
fit_solutions = solutions_df[solutions_df["is_converged"] == True]
fit_solutions

In [ ]:
solutions_df.query("total_score >= 5.57")

In [ ]:
solutionPSO_df = pd.read_csv("PSO_optimization_history_scaled_objective.csv")

In [ ]:
best_plane

In [ ]:
PSOfit_solutions = solutionPSO_df.query("final_score >= 3 and is_converged == True and n_pucks <= 9")
PSOfit_solutions

In [ ]:
solutionPSO_df.tail()


In [ ]:
no_dups_df.to_csv('cleaned_solutions_CAA291025.csv', index=False)

In [ ]:
no_dups_df.head()

In [ ]:
rc_plane = RCPlane(1.020966011428849, 1.124, 821.0, 4.0, 1.0, 3.0, 83.0, 75.0, 333.0, 4.0, 93.0)

In [ ]:
rc_plane.m2_payload

In [ ]:
rc_plane.m3_payload

In [ ]:
rc_plane.avionics

In [ ]:
rc_plane.propulsion

In [ ]:
rc_plane.m3

In [ ]:
rc_plane.m2